# Temporal Bleed: How Long Do Avalanches Persist Across Cycles?

The detection pipeline uses exponential decay temporal weights (`tau = 24 days`) centered
on the avalanche date. This means SAR acquisitions far from the event still contribute,
and avalanche debris from *neighboring* high-danger cycles will bleed into each other.

This notebook quantifies:
1. How wide the effective detection window is
2. How much signal from a nearby event bleeds into a given cycle
3. A toy example showing bleed between the Jan 25 and Feb 4, 2025 cycles

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.dates as mdates
from pathlib import Path

TAU_DAYS = 24  # from sarvalanche.weights.temporal

## 1. Theoretical Weight Decay

The temporal weight for a SAR acquisition at time offset `dt` from the avalanche date is:

$$w(\Delta t) \propto \exp\left(-\frac{|\Delta t|}{\tau}\right), \quad \tau = 24 \text{ days}$$

Normalized so all weights sum to 1. The key question: if an avalanche happened on day X,
how much weight does a cycle centered on day Y give to the SAR acquisitions near day X?

In [ ]:
# Continuous decay curve
dt = np.linspace(-80, 80, 1000)
w_unnorm = np.exp(-np.abs(dt) / TAU_DAYS)

fig, axes = plt.subplots(1, 2, figsize=(14, 5))

ax = axes[0]
ax.plot(dt, w_unnorm, 'k-', lw=2)
ax.axhline(0.5, color='tab:orange', ls='--', lw=1, label=f'50% at ±{TAU_DAYS * np.log(2):.0f}d')
ax.axhline(0.25, color='tab:red', ls='--', lw=1, label=f'25% at ±{TAU_DAYS * np.log(4):.0f}d')
ax.axhline(0.1, color='tab:purple', ls='--', lw=1, label=f'10% at ±{TAU_DAYS * np.log(10):.0f}d')
ax.set_xlabel('Days from avalanche date')
ax.set_ylabel('Unnormalized weight')
ax.set_title(f'Exponential decay (tau={TAU_DAYS}d)\nWeight drops to 50% at ±{TAU_DAYS * np.log(2):.0f} days')
ax.legend()
ax.grid(True, alpha=0.3)
ax.set_xlim(-80, 80)

# Cumulative weight within ± N days
ax = axes[1]
windows = np.arange(1, 61)
cum_frac = []
for d in windows:
    mask = np.abs(dt) <= d
    cum_frac.append(w_unnorm[mask].sum() / w_unnorm.sum())
ax.plot(windows, cum_frac, 'tab:blue', lw=2)
ax.axhline(0.5, color='gray', ls=':', lw=0.5)
ax.axhline(0.9, color='gray', ls=':', lw=0.5)
# Mark key thresholds
for target in [0.5, 0.75, 0.9, 0.95]:
    idx = np.searchsorted(cum_frac, target)
    if idx < len(windows):
        ax.annotate(f'{target:.0%} in ±{windows[idx]}d',
                    xy=(windows[idx], target), fontsize=9,
                    arrowprops=dict(arrowstyle='->', color='black'),
                    textcoords='offset points', xytext=(20, -10))
ax.set_xlabel('Window half-width (days)')
ax.set_ylabel('Fraction of total weight')
ax.set_title('Cumulative weight fraction within ±N days')
ax.grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

## 2. Bleed Between Two Cycles

If avalanche A happens on day 0 and cycle B is centered on day `d`, the SAR
acquisitions near day 0 get weight `exp(-|d|/tau)` relative to the peak in cycle B.
But in practice, SAR revisits every ~5-6 days per orbit, so the actual bleed
depends on which acquisitions fall near the event.

Simplified model: assume SAR acquisitions every 5 days. For a cycle centered at
offset `d`, what fraction of that cycle's total weight falls within ±6 days of day 0?

In [ ]:
# Simulate bleed for various separations between event and cycle center
sar_cadence = 5  # days between acquisitions
n_acq = 27  # typical number of acquisitions

separations = np.arange(0, 61)
bleed_fracs = []
event_day = 0  # avalanche happens on day 0
event_window = 6  # debris visible in SAR within ±6 days of event

for sep in separations:
    # Cycle centered at day `sep`
    acq_times = np.arange(-(n_acq // 2), n_acq // 2 + 1) * sar_cadence + sep
    dt_from_center = acq_times - sep
    weights = np.exp(-np.abs(dt_from_center) / TAU_DAYS)
    weights /= weights.sum()
    
    # Which acquisitions are within the event window?
    near_event = np.abs(acq_times - event_day) <= event_window
    bleed_fracs.append(weights[near_event].sum())

fig, ax = plt.subplots(figsize=(12, 5))
ax.plot(separations, np.array(bleed_fracs) * 100, 'tab:red', lw=2)
ax.fill_between(separations, 0, np.array(bleed_fracs) * 100, alpha=0.15, color='tab:red')

# Mark specific high-danger date separations
cycle_pairs = [
    ('Feb 4 ← Jan 25', 10),
    ('Feb 29 ← Feb 5', 24),
    ('Apr 1 ← Mar 13', 19),
    ('Feb 5 ← Jan 12', 24),
]
for label, sep in cycle_pairs:
    pct = bleed_fracs[sep] * 100
    ax.annotate(f'{label}\n{pct:.0f}%', xy=(sep, pct),
                fontsize=8, ha='center', va='bottom',
                arrowprops=dict(arrowstyle='->', color='black'),
                textcoords='offset points', xytext=(0, 15))

ax.set_xlabel('Days between avalanche event and cycle center')
ax.set_ylabel('% of cycle weight from event window (±6d)')
ax.set_title(f'Temporal bleed: how much of cycle B\'s weight falls near event A\n'
             f'(tau={TAU_DAYS}d, SAR cadence={sar_cadence}d, event window=±{event_window}d)')
ax.grid(True, alpha=0.3)
ax.set_xlim(0, 60)
ax.set_ylim(0, None)
plt.tight_layout()
plt.show()

## 3. Real Example: Feb 4, 2025 Cycle Seeing Jan 25 Debris

The Feb 4 cycle has actual SAR acquisitions on Jan 28 and Feb 2 — both within
a few days of a hypothetical Jan 25 event. Let's see exactly how much weight
those acquisitions carry.

In [ ]:
import xarray as xr

ds = xr.open_dataset(
    '/Users/zmhoppinen/Documents/sarvalanche/local/issw/high_danger_output/'
    'sarvalanche_runs/Banner_Summit_2025-02-04.nc'
)
wt = ds['w_temporal']
times = pd.DatetimeIndex(wt.time.values)
weights = wt.values
ds.close()

event_date = pd.Timestamp('2025-01-25')
center_date = pd.Timestamp('2025-02-04')

fig, ax = plt.subplots(figsize=(14, 5))

# Bar chart of weights
colors = []
for t in times:
    days_from_event = abs((t - event_date).days)
    if days_from_event <= 6:
        colors.append('tab:red')
    elif days_from_event <= 12:
        colors.append('tab:orange')
    else:
        colors.append('tab:blue')

ax.bar(times, weights, width=2, color=colors, edgecolor='black', lw=0.3)

# Mark dates
ax.axvline(event_date, color='red', ls='--', lw=2, label=f'Jan 25 event')
ax.axvline(center_date, color='black', ls='--', lw=2, label=f'Feb 4 cycle center')

# Shade the event window
ax.axvspan(event_date - pd.Timedelta(days=6), event_date + pd.Timedelta(days=6),
           alpha=0.1, color='red', label='±6d from Jan 25')

# Compute bleed
near_event = np.array([abs((t - event_date).days) <= 6 for t in times])
bleed_pct = weights[near_event].sum() * 100
near_event_12 = np.array([abs((t - event_date).days) <= 12 for t in times])
bleed_pct_12 = weights[near_event_12].sum() * 100

ax.set_xlabel('SAR acquisition date')
ax.set_ylabel('Temporal weight')
ax.set_title(
    f'Feb 4 cycle: {bleed_pct:.0f}% of weight within ±6d of Jan 25, '
    f'{bleed_pct_12:.0f}% within ±12d\n'
    f'Red bars = acquisitions that would see Jan 25 debris',
    fontsize=11,
)
ax.legend(loc='upper left', fontsize=9)
ax.xaxis.set_major_formatter(mdates.DateFormatter('%b %d'))
ax.grid(True, alpha=0.2, axis='y')
plt.xticks(rotation=45)
plt.tight_layout()
plt.show()

# Print the specific acquisitions
print(f'\nAcquisitions within ±6d of Jan 25 event:')
for t, w in zip(times, weights):
    days = (t - event_date).days
    if abs(days) <= 6:
        print(f'  {t.strftime("%Y-%m-%d")}  weight={w:.4f}  ({days:+d}d from event, '
              f'{(t - center_date).days:+d}d from center)')
print(f'  Total: {bleed_pct:.1f}% of cycle weight')

print(f'\nAcquisitions within ±12d of Jan 25 event:')
for t, w in zip(times, weights):
    days = (t - event_date).days
    if abs(days) <= 12:
        print(f'  {t.strftime("%Y-%m-%d")}  weight={w:.4f}  ({days:+d}d from event)')
print(f'  Total: {bleed_pct_12:.1f}% of cycle weight')

## 4. Pairwise Bleed Across All High-Danger Dates

For every pair of high-danger dates, compute how much of cycle B's weight
falls near event A. This reveals which cycles are most contaminated by neighbors.

In [ ]:
high_danger_dates = [
    '2022-12-12', '2023-03-13', '2023-04-01',
    '2024-01-12', '2024-02-05', '2024-02-29',
    '2024-12-29', '2025-02-04',
]

# For each pair: what fraction of cycle_j's weight falls within ±6d of event_i?
bleed_matrix = pd.DataFrame(0.0, index=high_danger_dates, columns=high_danger_dates)

for j, cycle_date in enumerate(high_danger_dates):
    cycle_center = pd.Timestamp(cycle_date)
    # Simulate SAR acquisitions (every 5 days, 27 total)
    acq_offsets = np.arange(-13, 14) * sar_cadence  # ±65 days
    acq_times_sim = [cycle_center + pd.Timedelta(days=int(d)) for d in acq_offsets]
    w = np.exp(-np.abs(acq_offsets) / TAU_DAYS)
    w /= w.sum()
    
    for i, event_date_str in enumerate(high_danger_dates):
        event = pd.Timestamp(event_date_str)
        near = np.array([abs((t - event).days) <= 6 for t in acq_times_sim])
        bleed_matrix.iloc[i, j] = w[near].sum() * 100

# Display as heatmap
fig, ax = plt.subplots(figsize=(10, 8))
im = ax.imshow(bleed_matrix.values, cmap='YlOrRd', vmin=0, vmax=40)
ax.set_xticks(range(len(high_danger_dates)))
ax.set_xticklabels(high_danger_dates, rotation=45, ha='right')
ax.set_yticks(range(len(high_danger_dates)))
ax.set_yticklabels(high_danger_dates)
ax.set_xlabel('Cycle center (detecting in)')
ax.set_ylabel('Event date (debris from)')
ax.set_title('Temporal bleed matrix (% of cycle weight within ±6d of event)\n'
             'Diagonal = self | Off-diagonal = cross-contamination')

# Annotate cells
for i in range(len(high_danger_dates)):
    for j in range(len(high_danger_dates)):
        val = bleed_matrix.iloc[i, j]
        if val > 1:
            color = 'white' if val > 20 else 'black'
            ax.text(j, i, f'{val:.0f}%', ha='center', va='center', fontsize=8, color=color)

plt.colorbar(im, ax=ax, label='% weight', shrink=0.8)
plt.tight_layout()
plt.show()

# Print significant cross-contamination
print('Significant cross-contamination (>5%):')
for i, ev in enumerate(high_danger_dates):
    for j, cy in enumerate(high_danger_dates):
        if i != j and bleed_matrix.iloc[i, j] > 5:
            print(f'  Debris from {ev} seen in {cy} cycle: {bleed_matrix.iloc[i, j]:.0f}%')

## 5. Summary

With `tau = 24 days`:

| Separation | Weight at offset | Bleed (±6d window) |
|-----------|-----------------|--------------------|
| 0 days (self) | 100% | ~30-35% |
| 10 days | 66% | ~20-25% |
| 17 days (tau * ln2) | 50% | ~15% |
| 24 days (1 tau) | 37% | ~10% |
| 48 days (2 tau) | 14% | ~3% |

**Practical implication**: Any two high-danger dates within ~3 weeks of each other will
have substantial cross-contamination. The Feb 4, 2025 cycle seeing Jan 25 debris is
expected — those acquisitions carry ~20-25% of the cycle's weight.

This is a feature, not a bug — the pipeline is designed to detect debris deposited
*near* the target date, not exclusively *on* it. But it means detections attributed
to one cycle may actually be from a neighboring event.